# Statystyka dla Data Analyst/Engineer — Moduł 4: Rozkłady prawdopodobieństwa

W Module 3 liczyliśmy prawdopodobieństwo pojedynczych zdarzeń. **Rozkład
prawdopodobieństwa** to matematyczny model opisujący *wszystkie możliwe* wyniki
losowego zjawiska naraz — i jak prawdopodobne jest każdy z nich. Poznamy cztery
najważniejsze rozkłady i nauczymy się posługiwać `scipy.stats`, żeby liczyć z nich
prawdopodobieństwa, zamiast tylko je symulować.

## Spis treści
1. [Zmienne losowe i rozkłady: PMF, PDF, CDF](#sec1)
2. [Rozkład jednostajny](#sec2)
3. [Rozkład dwumianowy](#sec3)
4. [Rozkład Poissona](#sec4)
5. [Rozkład normalny](#sec5)
6. [Reguła 68-95-99.7 i standaryzacja (z-score)](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Zmienne losowe i rozkłady: PMF, PDF, CDF

**Zmienna losowa** to liczba, której wartość zależy od wyniku losowego eksperymentu
(np. "liczba oczek na kostce", "wydatki losowego klienta"). Rozkłady dzielimy na dwa typy:

- **Dyskretne** — zmienna przyjmuje policzalny zbiór wartości (np. `0, 1, 2, 3...`).
  Opisuje je **PMF** (*Probability Mass Function*) — funkcja zwracająca `P(X = x)` dla
  każdej możliwej wartości `x`.
- **Ciągłe** — zmienna może przyjąć dowolną wartość z przedziału liczb rzeczywistych (np.
  wzrost, czas, wydatki). Opisuje je **PDF** (*Probability Density Function*) — ale
  UWAGA: `PDF(x)` to NIE jest `P(X = x)` (dla rozkładu ciągłego prawdopodobieństwo
  trafienia w dokładnie jeden punkt wynosi zero!). Prawdopodobieństwa dla ciągłych
  zmiennych liczymy jako **pole pod krzywą PDF** na jakimś przedziale.

Obu typów dotyczy **CDF** (*Cumulative Distribution Function*) — `CDF(x) = P(X <= x)`,
czyli prawdopodobieństwo, że zmienna losowa jest mniejsza lub równa `x`. `scipy.stats`
udostępnia dla każdego rozkładu metody `.pmf()`/`.pdf()`, `.cdf()`, oraz `.rvs()` (do
losowania próbek) i `.ppf()` (odwrotność CDF — "dla jakiego x, CDF(x) = p?").

<a id="sec2"></a>
## 2. Rozkład jednostajny

**Rozkład jednostajny** (*uniform*) oznacza, że każda wartość w danym przedziale jest
jednakowo prawdopodobna. Nasza kolumna `wiek` została właśnie tak wygenerowana
(`rng.integers(18, 70)`) — każdy wiek z przedziału 18-69 miał równe szanse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.figure(figsize=(8, 4))
plt.hist(klienci["wiek"], bins=20, edgecolor="white", density=True)
plt.xlabel("Wiek")
plt.ylabel("Gestosc")
plt.title("Wiek klientow -- przyblizenie rozkladu jednostajnego")
plt.show()


`scipy.stats.uniform` pozwala liczyć dokładne prawdopodobieństwa teoretyczne (a nie
tylko oglądać histogram). Domyślnie opisuje rozkład na przedziale `[0, 1]`, więc podajemy
`loc` (początek przedziału) i `scale` (długość przedziału):

In [ ]:
rozklad_wieku = stats.uniform(loc=18, scale=70 - 18)  # przedzial [18, 70)

# P(wiek <= 40) teoretycznie
print(f"P(wiek <= 40) teoretycznie: {rozklad_wieku.cdf(40):.3f}")
# to samo empirycznie, z naszych danych
print(f"P(wiek <= 40) empirycznie: {(klienci['wiek'] <= 40).mean():.3f}")


<a id="sec3"></a>
## 3. Rozkład dwumianowy

**Rozkład dwumianowy** (*binomial*) opisuje liczbę "sukcesów" w `n` niezależnych próbach,
z których każda ma to samo prawdopodobieństwo sukcesu `p` (np. `n` rzutów monetą, `n`
wysłanych maili z ofertą, gdzie każdy klient konwertuje z prawdopodobieństwem `p`). To
naturalny model dla naszej kolumny `konwersja`.

In [ ]:
# Jesli wyslemy oferte do 20 klientow z grupy A (P(konwersja) = 0.10 z zalozenia),
# ile z nich prawdopodobnie skonwertuje?
n_klientow = 20
p_konwersji = 0.10
rozklad_konwersji = stats.binom(n=n_klientow, p=p_konwersji)

# P(dokladnie 3 konwersje)
print(f"P(dokladnie 3 konwersje): {rozklad_konwersji.pmf(3):.3f}")
# P(co najwyzej 1 konwersja)
print(f"P(co najwyzej 1 konwersja): {rozklad_konwersji.cdf(1):.3f}")
# wartosc oczekiwana = n * p
print(f"Oczekiwana liczba konwersji: {rozklad_konwersji.mean():.1f}")


In [ ]:
x = np.arange(0, n_klientow + 1)
plt.figure(figsize=(8, 4))
plt.bar(x, rozklad_konwersji.pmf(x))
plt.xlabel("Liczba konwersji na 20 probek")
plt.ylabel("Prawdopodobienstwo (PMF)")
plt.title("Rozklad dwumianowy: n=20, p=0.10")
plt.show()


> 📊 **Rozkład dwumianowy to fundament testów A/B**
>
> Kiedy w Module 14 będziemy liczyć, ile obserwacji potrzeba do wiarygodnego testu A/B, w tle zawsze stoi właśnie rozkład dwumianowy — konwersja/brak konwersji to klasyczna próba Bernoulliego, a suma wielu takich prób to rozkład dwumianowy.

<a id="sec4"></a>
## 4. Rozkład Poissona

**Rozkład Poissona** opisuje liczbę zdarzeń zachodzących w ustalonym przedziale czasu
(lub przestrzeni), gdy zdarzenia występują ze znaną średnią częstością `lambda`, niezależnie
od siebie. Typowe przykłady: liczba zamówień na godzinę, liczba awarii serwera dziennie,
liczba klientów wchodzących do sklepu na minutę.

In [ ]:
# Zalozmy, ze sklep dostaje srednio 4 zamowienia na godzine (lambda=4)
lambda_zamowien = 4
rozklad_zamowien = stats.poisson(mu=lambda_zamowien)

print(f"P(dokladnie 4 zamowienia w godzine): {rozklad_zamowien.pmf(4):.3f}")
print(f"P(0 zamowien w godzine): {rozklad_zamowien.pmf(0):.3f}")
print(f"P(wiecej niz 8 zamowien w godzine): {1 - rozklad_zamowien.cdf(8):.3f}")

x = np.arange(0, 15)
plt.figure(figsize=(8, 4))
plt.bar(x, rozklad_zamowien.pmf(x))
plt.xlabel("Liczba zamowien w ciagu godziny")
plt.ylabel("Prawdopodobienstwo (PMF)")
plt.title("Rozklad Poissona: lambda=4")
plt.show()


> ⚠️ **Poisson zakłada stałe tempo i niezależność zdarzeń**
>
> Rozkład Poissona zakłada, że zdarzenia zachodzą w stałym, przeciętnym tempie i niezależnie od siebie (jedno zamówienie nie zwiększa ani nie zmniejsza szansy na kolejne). W realnym sklepie to uproszczenie — np. promocja może chwilowo podbić tempo zamówień — ale jako punkt wyjścia model Poissona jest bardzo użyteczny i zaskakująco dobrze się sprawdza dla wielu zjawisk "rzadkich zdarzeń".

<a id="sec5"></a>
## 5. Rozkład normalny

**Rozkład normalny** (Gaussa, "dzwonowy") to najważniejszy rozkład ciągły w statystyce —
symetryczny, w pełni opisany dwoma parametrami: średnią `mu` (środek, szczyt krzywej) i
odchyleniem standardowym `sigma` (jak bardzo dane są rozrzucone wokół średniej). Nasza
kolumna `wydatki_przed` została wygenerowana właśnie z rozkładu normalnego
(`N(220, 40)`).

In [ ]:
rozklad_wydatkow_przed = stats.norm(loc=220, scale=40)

plt.figure(figsize=(8, 4))
plt.hist(klienci["wydatki_przed"], bins=30, density=True, edgecolor="white", alpha=0.6, label="Dane (histogram)")

x = np.linspace(klienci["wydatki_przed"].min(), klienci["wydatki_przed"].max(), 200)
plt.plot(x, rozklad_wydatkow_przed.pdf(x), color="red", linewidth=2, label="Teoretyczny PDF N(220, 40)")
plt.xlabel("Wydatki przed programem lojalnosciowym")
plt.ylabel("Gestosc")
plt.title("Wydatki 'przed' na tle teoretycznego rozkladu normalnego")
plt.legend()
plt.show()


Histogram naszych danych niemal idealnie pokrywa się z teoretyczną krzywą — bo
faktycznie tak wygenerowaliśmy te dane. W praktyce nigdy nie znasz "prawdziwych"
parametrów — szacujesz je z próbki (`klienci['wydatki_przed'].mean()` i `.std()`), co
robi dokładnie to samo, co Moduł 1.

<a id="sec6"></a>
## 6. Reguła 68-95-99.7 i standaryzacja (z-score)

Dla rozkładu normalnego obowiązuje wygodna reguła kciuka:
- ok. **68%** obserwacji leży w przedziale `mu ± 1*sigma`,
- ok. **95%** leży w przedziale `mu ± 2*sigma`,
- ok. **99.7%** leży w przedziale `mu ± 3*sigma`.

**Standaryzacja** (z-score) przekształca dowolną wartość `x` na liczbę odchyleń
standardowych od średniej: `z = (x - mu) / sigma`. Pozwala porównywać wartości z różnych
rozkładów na wspólnej skali.

In [ ]:
mu = klienci["wydatki_przed"].mean()
sigma = klienci["wydatki_przed"].std(ddof=0)  # tu uzywamy parametrow populacji (ddof=0),
                                                # bo porownujemy z definicja teoretyczna

for k in [1, 2, 3]:
    udzial = ((klienci["wydatki_przed"] >= mu - k * sigma) & (klienci["wydatki_przed"] <= mu + k * sigma)).mean()
    print(f"Udzial w przedziale mu +/- {k} sigma: {udzial:.3f}")


In [ ]:
# z-score jednego konkretnego klienta
przykladowy_klient = klienci.iloc[0]
z = (przykladowy_klient["wydatki_przed"] - mu) / sigma
print(f"Wydatki klienta: {przykladowy_klient['wydatki_przed']:.2f}")
print(f"Z-score: {z:.2f} odchylen standardowych od sredniej")

# scipy.stats.zscore robi to samo dla calej kolumny naraz
z_wszystkie = stats.zscore(klienci["wydatki_przed"])
print(z_wszystkie[:5])


> 📊 **Z-score to prekursor testów z Modułu 7-8**
>
> Statystyka testowa w teście z (a w praktyce często też w teście t) to w gruncie rzeczy z-score — "o ile odchyleń standardowych obserwowany wynik odbiega od tego, czego oczekiwalibyśmy pod hipotezą zerową". Ten sam wzór wróci w Module 7.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- różnicę między PMF (rozkłady dyskretne), PDF (rozkłady ciągłe) i CDF (oba typy),
- rozkład jednostajny (nasza kolumna `wiek`),
- rozkład dwumianowy — liczba sukcesów w `n` próbach (konwersje),
- rozkład Poissona — liczba zdarzeń w ustalonym czasie (zamówienia na godzinę),
- rozkład normalny — najważniejszy rozkład ciągły (`wydatki_przed`),
- regułę 68-95-99.7 i standaryzację (z-score).

> 📝 **Ćwiczenie 1: Prawdopodobieństwo z rozkładu dwumianowego**
>
> Grupa B ma `P(konwersja) = 0.20`. Jeśli wyślemy ofertę do 15 klientów z grupy B, jakie jest prawdopodobieństwo, że skonwertuje DOKŁADNIE 5 z nich? A jakie, że skonwertuje CO NAJMNIEJ 5?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
rozklad_b = stats.binom(n=15, p=0.20)
print(f"Dokladnie 5: {rozklad_b.pmf(5):.3f}")
print(f"Co najmniej 5: {1 - rozklad_b.cdf(4):.3f}")
```

'Co najmniej 5' to dopelnienie 'co najwyzej 4' -- 1 - CDF(4), a nie 1 - CDF(5), bo CDF(4) juz wlacza wszystkie wartosci <= 4.
</details>

> 📝 **Ćwiczenie 2: Poisson: prawdopodobieństwo zera zdarzeń**
>
> Serwer padał średnio 0.5 razy dziennie (`lambda=0.5`). Jakie jest prawdopodobieństwo całego tygodnia (7 dni) BEZ ŻADNEJ awarii? Podpowiedź: przelicz `lambda` na cały tydzień (`lambda * 7`), zanim policzysz `pmf(0)`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
lambda_tydzien = 0.5 * 7
rozklad_awarii = stats.poisson(mu=lambda_tydzien)
print(rozklad_awarii.pmf(0))
```
</details>

> 📝 **Ćwiczenie 3: Standaryzacja satysfakcji**
>
> Policz z-score kolumny `satysfakcja` (użyj `stats.zscore`). Ile obserwacji ma `|z| > 2` (czyli leży dalej niż 2 odchylenia standardowe od średniej)? Porównaj tę liczbę z regułą 68-95-99.7 z sekcji 6 -- czy jest zbliżona do oczekiwanych ~5%?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
z_satysfakcja = stats.zscore(klienci["satysfakcja"])
udzial_odstajacych = (np.abs(z_satysfakcja) > 2).mean()
print(f"Udzial z |z| > 2: {udzial_odstajacych:.3f}")
print("Oczekiwane wg reguly 68-95-99.7: okolo 1 - 0.95 = 0.05")
```
</details>

> 📝 **Ćwiczenie 4: Percentyl z rozkładu normalnego**
>
> Użyj `stats.norm(loc=mu, scale=sigma).ppf(0.90)` (gdzie `mu`/`sigma` to średnia/odchylenie std. `wydatki_przed` z sekcji 6), żeby policzyć TEORETYCZNY 90. percentyl wydatków 'przed'. Porównaj z EMPIRYCZNYM percentylem `klienci['wydatki_przed'].quantile(0.90)` z Modułu 1 -- czy są bliskie?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
rozklad = stats.norm(loc=mu, scale=sigma)
percentyl_teoretyczny = rozklad.ppf(0.90)
percentyl_empiryczny = klienci["wydatki_przed"].quantile(0.90)

print(f"Teoretyczny (z rozkladu normalnego): {percentyl_teoretyczny:.2f}")
print(f"Empiryczny (z danych): {percentyl_empiryczny:.2f}")
```

Powinny wyjsc bardzo blisko siebie, bo wydatki_przed faktycznie pochodza z rozkladu normalnego -- 'ppf' to dokladnie odwrotnosc 'cdf' z sekcji 1.
</details>

> 🔥 **Wyzwanie: symulacja kontra teoria**
>
> Wylosuj (`stats.norm(loc=220, scale=40).rvs(size=10000, random_state=42)`) 10 000 wartości z teoretycznego rozkładu normalnego. Narysuj na jednym wykresie histogram wylosowanych danych ORAZ histogram prawdziwej kolumny `wydatki_przed` (oba z `density=True`, żeby były porównywalne mimo różnej liczby obserwacji). Czy kształty są zbliżone, mimo że jeden zbiór ma 10 000 punktów, a drugi tylko 400?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
symulowane = stats.norm(loc=220, scale=40).rvs(size=10_000, random_state=42)

plt.figure(figsize=(8, 4))
plt.hist(symulowane, bins=40, density=True, alpha=0.5, label="Symulacja (n=10000)")
plt.hist(klienci["wydatki_przed"], bins=30, density=True, alpha=0.5, label="Dane (n=400)")
plt.xlabel("Wydatki przed")
plt.ylabel("Gestosc")
plt.legend()
plt.title("Symulacja vs prawdziwe dane")
plt.show()
```
</details>

## Co dalej?

W **Module 5** poznamy jedno z najważniejszych twierdzeń całej statystyki: **Centralne
Twierdzenie Graniczne**. Zobaczymy, dlaczego rozkład *średnich z próbek* zbliża się do
rozkładu normalnego, nawet gdy oryginalne dane (jak nasze prawoskośne `wydatki_miesieczne`)
wcale normalne nie są — i dlaczego to jest fundamentem niemal całej statystyki
wnioskującej.